# Orbit Viewer — 99942 Apophis
Visualizador de planos orbitales inspirado en el **JPL SBDB View Orbit Plane (VOP)**, construido sobre `pymcel` y `Plotly`.

Muestra las órbitas de Mercurio, Venus, Tierra, Marte y **Apophis**, con:
- **Tick marks** (picket-fence) que ilustran la inclinación respecto a la eclíptica.
- **Posición actual** de cada cuerpo al **2026-04-22 00:00 UTC**.
- **Línea de nodos** (amarilla) y **normal al plano orbital** (verde) de Apophis.
- Banda sombreada entre la eclíptica y el plano orbital de Apophis.

In [1]:
# Instalar dependencias (omitir si ya están instaladas)
import sys
!{sys.executable} -m pip install -q pymcel plotly numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 60.4 MB/s eta 0:00:00


## Imports

In [81]:
import numpy as np
import plotly.graph_objects as go
from dataclasses import dataclass
import pymcel as pc

@dataclass
class OrbitElements:
    a: float      # Semimajor axis [AU]
    e: float      # Eccentricity
    i: float      # Inclination [deg]
    raan: float   # Longitude of Ascending Node [deg]
    argp: float   # Argument of Perihelion [deg]
    M0: float     # Mean Anomaly at epoch [deg]
    epoch: float  # Epoch [days]
    mu: float     # Gravitational parameter

def elements_to_cartesian(el, t):
    """Propagates elements using pymcel (pc) and returns position vector."""
    n = np.sqrt(el.mu / el.a**3)
    M_t = np.radians(el.M0) + n * (t - el.epoch)
    kepler_list = [el.a, el.e, np.radians(el.i), np.radians(el.raan), np.radians(el.argp), M_t]
    estado = pc.elementos_a_estado(el.mu, kepler_list)
    return np.array(estado[:3])

def propagate_trajectory(el, n_pts=500):
    """Genera una trayectoria completa para una órbita."""
    T = 2 * np.pi * np.sqrt(el.a**3 / el.mu)
    times = np.linspace(0, T, n_pts)
    return np.array([elements_to_cartesian(el, t) for t in times])

def make_orbit_viewer(body_name, body_positions, body_elements, body_color, body_epoch_pos, reference_bodies, title, n_ticks=60, show_geometry=True, **kwargs):
    fig = go.Figure()
    # Sol
    fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers+text", text=["Sol"], textposition="top center", marker=dict(size=8, color="yellow"), name="Sol"))

    # Punto Vernal (Eje X / Aries)
    fig.add_trace(go.Scatter3d(x=[0, 1.5], y=[0, 0], z=[0, 0], mode="lines+text", text=["", "♈ Punto Vernal"], line=dict(color="magenta", width=5), name="Dirección Vernal (X)"))

    # Cuerpos de referencia (Ajustada opacidad para visibilidad)
    for rb in reference_bodies:
        pos = rb["positions"]
        fig.add_trace(go.Scatter3d(x=pos[:,0], y=pos[:,1], z=pos[:,2], mode="lines", line=dict(color=rb["color"], width=2, dash="dot"), name=rb["name"], opacity=0.6))
        fig.add_trace(go.Scatter3d(x=[rb["epoch_pos"][0]], y=[rb["epoch_pos"][1]], z=[rb["epoch_pos"][2]], mode="markers", marker=dict(size=5, color=rb["color"]), showlegend=False))

    # Órbita Apophis
    fig.add_trace(go.Scatter3d(x=body_positions[:,0], y=body_positions[:,1], z=body_positions[:,2], mode="lines", line=dict(color=body_color, width=4), name=body_name))
    fig.add_trace(go.Scatter3d(x=[body_epoch_pos[0]], y=[body_epoch_pos[1]], z=[body_epoch_pos[2]], mode="markers+text", text=[body_name], textposition="top center", marker=dict(size=6, color=body_color), showlegend=False))

    if show_geometry:
        el = body_elements
        p_pos = elements_to_cartesian(OrbitElements(el.a, el.e, el.i, el.raan, el.argp, 0, 0, el.mu), 0)
        a_pos = elements_to_cartesian(OrbitElements(el.a, el.e, el.i, el.raan, el.argp, 180, 0, el.mu), 0)

        fig.add_trace(go.Scatter3d(x=[p_pos[0], a_pos[0]], y=[p_pos[1], a_pos[1]], z=[p_pos[2], a_pos[2]], mode="lines", line=dict(color="red", width=2, dash="dash"), name="Eje de Ápsides"))
        fig.add_trace(go.Scatter3d(x=[p_pos[0]], y=[p_pos[1]], z=[p_pos[2]], mode="markers+text", text=["Periapsis"], marker=dict(size=4, color="red"), showlegend=False))
        fig.add_trace(go.Scatter3d(x=[a_pos[0]], y=[a_pos[1]], z=[a_pos[2]], mode="markers+text", text=["Apoapsis"], marker=dict(size=4, color="red"), showlegend=False))

        raan_rad = np.radians(el.raan)
        node_asc = np.array([np.cos(raan_rad), np.sin(raan_rad), 0]) * el.a
        node_desc = -node_asc
        fig.add_trace(go.Scatter3d(x=[node_asc[0], node_desc[0]], y=[node_asc[1], node_desc[1]], z=[0, 0], mode="lines", line=dict(color="cyan", width=3), name="Línea de Nodos"))
        fig.add_trace(go.Scatter3d(x=[node_asc[0]], y=[node_asc[1]], z=[0], mode="text", text=["☊ Nodo Asc."], textfont=dict(color="cyan"), showlegend=False))

        fig.add_trace(go.Scatter3d(x=[0, body_epoch_pos[0]], y=[0, body_epoch_pos[1]], z=[0, body_epoch_pos[2]], mode="lines", line=dict(color="yellow", width=2), name="Radio Vector (Anomalía V.)"))

    tick_indices = np.linspace(0, len(body_positions)-1, n_ticks, dtype=int)
    for idx in tick_indices:
        p = body_positions[idx]
        fig.add_trace(go.Scatter3d(x=[p[0], p[0]], y=[p[1], p[1]], z=[0, p[2]], mode="lines", line=dict(color="gray", width=1), opacity=0.3, showlegend=False))

    fig.update_layout(title=title, scene=dict(xaxis_title="X (AU)", yaxis_title="Y (AU)", zaxis_title="Z (AU)", aspectmode="data", bgcolor="black"), template="plotly_dark")
    return fig

## Constantes y elementos orbitales (J2000)

In [78]:
# Parámetro gravitacional del Sol  [AU³/día²]
MU_SUN = 2.9591220828559093e-4

# Días desde J2000 (2000-Jan-1.5) hasta 2026-Apr-22.0
T_CURRENT = 9608.5

# ─── Elementos orbitales en J2000  (a [AU], ángulos en grados, M0 en grados) ───
MERCURY = OrbitElements(a=0.38710, e=0.20563, i=7.005,  raan=48.331,  argp=29.124,  M0=174.796, epoch=0.0, mu=MU_SUN)
VENUS   = OrbitElements(a=0.72333, e=0.00677, i=3.395,  raan=76.680,  argp=55.186,  M0=50.114,  epoch=0.0, mu=MU_SUN)
EARTH   = OrbitElements(a=1.00000, e=0.01671, i=0.000,  raan=-11.261, argp=102.937, M0=8.789,   epoch=0.0, mu=MU_SUN)
MARS    = OrbitElements(a=1.52366, e=0.09341, i=1.850,  raan=49.562,  argp=-73.517, M0=19.402,  epoch=0.0, mu=MU_SUN)
APOPHIS = OrbitElements(a=0.92238, e=0.19117, i=3.341,  raan=203.900, argp=126.673, M0=122.0,   epoch=0.0, mu=MU_SUN)

## Propagación de órbitas

In [79]:
def elements_to_cartesian(el, t):
    """Propagates elements using pymcel with correct argument order (mu, elements)."""
    n = np.sqrt(el.mu / el.a**3)
    M_t = np.radians(el.M0) + n * (t - el.epoch)
    elementos_kepler = [el.a, el.e, np.radians(el.i), np.radians(el.raan), np.radians(el.argp), M_t]

    # El orden oficial de pymcel es (parametro_gravitacional, lista_elementos)
    # La función devuelve un solo array: [x, y, z, vx, vy, vz]
    estado = pc.elementos_a_estado(el.mu, elementos_kepler)

    # Extraemos los primeros 3 elementos (posición r)
    return np.array(estado[:3])

def propagate_body(elements):
    positions = propagate_trajectory(elements)
    epoch_pos = elements_to_cartesian(elements, T_CURRENT)
    return positions, epoch_pos

print('Propagando órbitas con motor corregido (pymcel)...')
mercury_pos, mercury_now = propagate_body(MERCURY)
venus_pos,   venus_now   = propagate_body(VENUS)
earth_pos,   earth_now   = propagate_body(EARTH)
mars_pos,    mars_now    = propagate_body(MARS)
apophis_pos, apophis_now = propagate_body(APOPHIS)
print('¡Propagación completada exitosamente!')

Propagando órbitas con motor corregido (pymcel)...
¡Propagación completada exitosamente!


## Visualización interactiva
La figura es un `plotly.graph_objects.Figure` embebido en el notebook.  
Usa los controles de Plotly para rotar, hacer zoom y activar/desactivar trazas.

In [80]:
reference_bodies = [
    {"name": "Mercurio", "positions": mercury_pos, "color": "#ff80c0", "epoch_pos": mercury_now},
    {"name": "Venus",    "positions": venus_pos,   "color": "#9040d0", "epoch_pos": venus_now},
    {"name": "Tierra",   "positions": earth_pos,   "color": "#40a0ff", "epoch_pos": earth_now},
    {"name": "Marte",    "positions": mars_pos,    "color": "#ff5500", "epoch_pos": mars_now},
]

fig = make_orbit_viewer(
    body_name="99942 Apophis",
    body_positions=apophis_pos,
    body_elements=APOPHIS,
    body_color="#e0e0e0",
    body_epoch_pos=apophis_now,
    reference_bodies=reference_bodies,
    epoch_label="2026-04-22 00:00:00 UTC",
    title="Orbit Viewer — 99942 Apophis",
    n_ticks=90,
    show_wedge=True,
)
fig.show()

### 🕹️ Navegador Temporal Interactivo (Estilo JPL VOP)
Esta sección permite seleccionar fechas específicas o animar el sistema solar interior para observar la evolución de la órbita de Apophis.

In [74]:
import datetime
from ipywidgets import interact, widgets, Layout

def date_to_j2000(date_str):
    """Convierte una fecha YYYY-MM-DD a días desde J2000 (2000-01-01 12:00 UTC)."""
    dt = datetime.datetime.strptime(date_str, "%Y-%m-%d")
    j2000_epoch = datetime.datetime(2000, 1, 1, 12, 0)
    delta = dt - j2000_epoch
    return delta.total_seconds() / 86400.0

def update_viewer(fecha):
    t_target = date_to_j2000(str(fecha))

    # Recalcular posiciones actuales para la fecha seleccionada
    m_now = elements_to_cartesian(MERCURY, t_target)
    v_now = elements_to_cartesian(VENUS, t_target)
    e_now = elements_to_cartesian(EARTH, t_target)
    ma_now = elements_to_cartesian(MARS, t_target)
    a_now = elements_to_cartesian(APOPHIS, t_target)

    ref_bodies = [
        {"name": "Mercurio", "positions": mercury_pos, "color": "#ff80c0", "epoch_pos": m_now},
        {"name": "Venus",    "positions": venus_pos,   "color": "#9040d0", "epoch_pos": v_now},
        {"name": "Tierra",   "positions": earth_pos,   "color": "#40a0ff", "epoch_pos": e_now},
        {"name": "Marte",    "positions": mars_pos,    "color": "#ff5500", "epoch_pos": ma_now},
    ]

    fig_interactive = make_orbit_viewer(
        body_name="99942 Apophis",
        body_positions=apophis_pos,
        body_elements=APOPHIS,
        body_color="#e0e0e0",
        body_epoch_pos=a_now,
        reference_bodies=ref_bodies,
        title=f"Orbit Viewer — Apophis | Fecha: {fecha}",
        n_ticks=60
    )

    fig_interactive.show()

# Widget de selección de fecha
interact(
    update_viewer,
    fecha=widgets.DatePicker(
        description='Escoger Fecha',
        #predeterminada la de mayor acercamiento de apophis
        value=datetime.date(2029, 4, 13),
        layout=Layout(width='300px')
    )
)

interactive(children=(DatePicker(value=datetime.date(2029, 4, 13), description='Escoger Fecha', layout=Layout(…

<function __main__.update_viewer(fecha)>

### 1. Selección de Fecha de Observación
Ejecuta esta celda y selecciona la fecha en el calendario. El valor se guardará automáticamente.

In [75]:
import datetime
from ipywidgets import widgets

# Widget global para la fecha
date_picker = widgets.DatePicker(
    description='Fecha de interés',
    value=datetime.date(2029, 4, 13),
    disabled=False
)

display(date_picker)

DatePicker(value=datetime.date(2029, 4, 13), description='Fecha de interés')

### 2. Generar Visualización
Una vez seleccionada la fecha arriba, ejecuta esta celda para actualizar el Orbit Viewer.

In [76]:
selected_date = date_picker.value.strftime('%Y-%m-%d')
t_target = date_to_j2000(selected_date)

print(f"Generando Orbit Viewer con Geometría para: {selected_date}")

# Recalcular posiciones actuales
m_now = elements_to_cartesian(MERCURY, t_target)
v_now = elements_to_cartesian(VENUS, t_target)
e_now = elements_to_cartesian(EARTH, t_target)
ma_now = elements_to_cartesian(MARS, t_target)
a_now = elements_to_cartesian(APOPHIS, t_target)

ref_bodies = [
    {"name": "Mercurio", "positions": mercury_pos, "color": "#ff80c0", "epoch_pos": m_now},
    {"name": "Venus",    "positions": venus_pos,   "color": "#9040d0", "epoch_pos": v_now},
    {"name": "Tierra",   "positions": earth_pos,   "color": "#40a0ff", "epoch_pos": e_now},
    {"name": "Marte",    "positions": mars_pos,    "color": "#ff5500", "epoch_pos": ma_now},
]

fig_final = make_orbit_viewer(
    body_name="99942 Apophis",
    body_positions=apophis_pos,
    body_elements=APOPHIS,
    body_color="#ffffff",
    body_epoch_pos=a_now,
    reference_bodies=ref_bodies,
    title=f"Orbit Viewer Geométrico — Apophis | Fecha: {selected_date}",
    n_ticks=100,
    show_geometry=True
)

fig_final.show()

Generando Orbit Viewer con Geometría para: 2029-04-13


### 🎬 Animación Temporal: Evolución de la Órbita
Esta celda genera una animación de Plotly que muestra el movimiento de los cuerpos durante un periodo de tiempo. Útil para visualizar el encuentro de 2029.

In [82]:
import pandas as pd

# Configuración de la animación
fecha_inicio = "2028-10-01"
num_frames = 60
dias_por_frame = 5
t_start = date_to_j2000(fecha_inicio)

frames = []
dates = []

print("Preparando frames de animación...")

for f in range(num_frames):
    t_current = t_start + f * dias_por_frame
    current_date = (datetime.datetime(2000, 1, 1, 12, 0) + datetime.timedelta(days=t_current)).strftime('%Y-%m-%d')
    dates.append(current_date)

    # Calcular posiciones
    m_p = elements_to_cartesian(MERCURY, t_current)
    v_p = elements_to_cartesian(VENUS, t_current)
    e_p = elements_to_cartesian(EARTH, t_current)
    ma_p = elements_to_cartesian(MARS, t_current)
    a_p = elements_to_cartesian(APOPHIS, t_current)

    # Crear el frame
    frame_data = [
        go.Scatter3d(x=[0], y=[0], z=[0], mode="markers", marker=dict(size=10, color="yellow"), name="Sol"),
        go.Scatter3d(x=[m_p[0]], y=[m_p[1]], z=[m_p[2]], mode="markers", marker=dict(size=4, color="#ff80c0"), name="Mercurio"),
        go.Scatter3d(x=[v_p[0]], y=[v_p[1]], z=[v_p[2]], mode="markers", marker=dict(size=5, color="#9040d0"), name="Venus"),
        go.Scatter3d(x=[e_p[0]], y=[e_p[1]], z=[e_p[2]], mode="markers", marker=dict(size=6, color="#40a0ff"), name="Tierra"),
        go.Scatter3d(x=[ma_p[0]], y=[ma_p[1]], z=[ma_p[2]], mode="markers", marker=dict(size=5, color="#ff5500"), name="Marte"),
        go.Scatter3d(x=[a_p[0]], y=[a_p[1]], z=[a_p[2]], mode="markers+text", text=["Apophis"], marker=dict(size=6, color="white"), name="Apophis")
    ]
    frames.append(go.Frame(data=frame_data, name=current_date))

# Crear figura base con órbitas estáticas
fig_anim = go.Figure(
    data=frames[0].data,
    layout=go.Layout(
        title="Animación Orbital: Encuentro Apophis 2029",
        scene=dict(
            xaxis=dict(range=[-2, 2]), yaxis=dict(range=[-2, 2]), zaxis=dict(range=[-1, 1]),
            aspectmode='data', bgcolor="black"
        ),
        updatemenus=[{
            "buttons": [
                {
                    "args": [None, {"frame": {"duration": 50, "redraw": True}, "fromcurrent": True}],
                    "label": "Play", "method": "animate"
                },
                {
                    "args": [[None], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate", "transition": {"duration": 0}}],
                    "label": "Pause", "method": "animate"
                }
            ],
            "type": "buttons", "showactive": False, "x": 0.1, "y": 0
        }],
        sliders=[{
            "steps": [{"args": [[f.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}], "label": f.name, "method": "animate"} for f in frames],
            "transition": {"duration": 0}, "x": 0.1, "len": 0.9
        }]
    ),
    frames=frames
)

fig_anim.update_layout(template="plotly_dark")
fig_anim.show()

Preparando frames de animación...
